In [1]:
import numpy as np
from qdk_chemistry.data import Element, Structure, MajoranaMapping
from qdk_chemistry.algorithms import create
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

coords_ang = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 0.740848],
    ]
)
structure = Structure(coords_ang, [Element.H, Element.H])

scf = create("scf_solver")
scf_energy, wavefunction = scf.run(
    structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g"
)

# 2) Electronic Hamiltonian
ham_constructor = create("hamiltonian_constructor")
hamiltonian = ham_constructor.run(wavefunction.get_orbitals())

# 3) Qubit Hamiltonian
mapper = create("qubit_mapper")
n_spin_orbitals = hamiltonian.get_orbitals().get_num_molecular_orbitals() * 2
qubit_hamiltonian = mapper.run(hamiltonian, MajoranaMapping.jordan_wigner(n_spin_orbitals))

# 4) Ground-state energy of the qubit Hamiltonian
solver = create("qubit_hamiltonian_solver")
qubit_energy, ground_state = solver.run(qubit_hamiltonian)
qubit_energy = qubit_energy

In [2]:
import scipy
from qdk_chemistry.utils.pauli_commutation import commutator_bound_first_order
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.time_evolution.trotter import Trotter

from qdk_chemistry.algorithms.hamiltonian_unitary_builder.time_evolution.trotter_error import trotter_steps_commutator
import math

_PAULI = {
    "I": np.eye(2, dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def _pauli_matrix(pauli_map: dict[int, str], num_qubits: int) -> np.ndarray:
    """Build the full 2^n x 2^n matrix for a sparse Pauli map {qubit_idx: 'X'/'Y'/'Z'}."""
    mat = np.array([[1.0]], dtype=complex)
    for q in range(num_qubits):
        mat = np.kron(mat, _PAULI[pauli_map.get(q, "I")])
    return mat

def numerical_error_sweep(hamiltonian, t=0.1, num_divisions=1):
    num_qubits = hamiltonian.num_qubits

    builder = Trotter(num_divisions=num_divisions, time=t)
    unitary = builder.run(hamiltonian)
    container = unitary.get_container()

    # Build Trotter unitary matrix
    dim = 2**num_qubits
    u_trot = np.eye(dim, dtype=complex)
    for term in container.step_terms:
        pauli_mat = _pauli_matrix(term.pauli_term, num_qubits)
        u_trot = scipy.linalg.expm(-1j * term.angle * pauli_mat) @ u_trot

    # Exact unitary
    hamiltonian_matrix = hamiltonian.to_matrix()
    u_exact = scipy.linalg.expm(-1j * t * hamiltonian_matrix)

    # State-specific error: ||U_trot|psi0> - U_exact|psi0>||_2
    psi_trot = u_trot @ ground_state
    psi_exact = u_exact @ ground_state
    ground_state_error = np.linalg.norm(psi_trot - psi_exact)
    operator_error = np.linalg.norm(u_trot - u_exact)
    return ground_state_error, operator_error

t = math.pi/8
print(f"t={t}, qubit_hamiltonian.schatten_norm={qubit_hamiltonian.schatten_norm}")
target_accuracy = 1e-6
order = 1
for num_phase_bit in range(1, 6):
    scaled_t = t * 2**(num_phase_bit - 1)
    step = trotter_steps_commutator(qubit_hamiltonian, scaled_t, target_accuracy, order=order)
    for num_divisions in range(1, step, math.ceil(step/4)):
        ground_state_error, operator_error = numerical_error_sweep(qubit_hamiltonian, t=scaled_t, num_divisions=num_divisions)
        print(
            f"t={scaled_t}, num_divisions={num_divisions}, operator_error={operator_error}, ground_state_error={ground_state_error}"
        )


t=0.39269908169872414, qubit_hamiltonian.schatten_norm=3.154411119206714
t=0.39269908169872414, num_divisions=1, operator_error=2.124676950909355, ground_state_error=1.041726474598244
t=0.39269908169872414, num_divisions=8837, operator_error=1.5832029370047571, ground_state_error=0.8541460673394995
t=0.39269908169872414, num_divisions=17673, operator_error=1.5832098306561433, ground_state_error=0.8541355132850786
t=0.39269908169872414, num_divisions=26509, operator_error=1.5832121298255732, ground_state_error=0.8541319950037316
t=0.7853981633974483, num_divisions=1, operator_error=3.8718919261895555, ground_state_error=1.776016015288936
t=0.7853981633974483, num_divisions=35345, operator_error=3.0189588487748313, ground_state_error=1.5446447729709334
t=0.7853981633974483, num_divisions=70689, operator_error=3.0189627792886666, ground_state_error=1.5446410653024703
t=0.7853981633974483, num_divisions=106033, operator_error=3.0189640896143257, ground_state_error=1.5446398293863832
t=1.57

## Old code for resource estimate the circuit
This cell takes about 40 secs

In [3]:
from utils.qpe_utils import compute_evolution_time
from qdk_chemistry.data import AlgorithmRef
from qdk_chemistry.algorithms.state_preparation import identity_state_prep
import pandas as pd


def run_trotter(m_precision, trotter_order, target_accuracy=1e-2):
    evolution_time = compute_evolution_time(qubit_hamiltonian, num_bits=m_precision)
    unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "trotter", time=evolution_time, order=trotter_order, power_strategy="rescale", target_accuracy=target_accuracy)
    circuit_mapper = AlgorithmRef("controlled_circuit_mapper", "pauli_sequence")
    circuit_builder = create(
        "qpe_circuit_builder", 
        "qdk_iterative",
        num_bits=m_precision,
        unitary_builder=unitary_builder,
        controlled_circuit_mapper=circuit_mapper,
    )
    state_prep = identity_state_prep(num_qubits=qubit_hamiltonian.num_qubits)
    iqpe_iter_circuits = circuit_builder.run(
        state_preparation=state_prep,
        qubit_hamiltonian=qubit_hamiltonian,
    )
    largest_circuit = iqpe_iter_circuits[0]
    result = largest_circuit.estimate()
    logical_estimate = result["logicalCounts"]

    return logical_estimate, target_accuracy, largest_circuit

rows = []
for m_precision in [6, 8, 10]:
    for trotter_order in [1, 2]:
        logical_estimate, target_accuracy, largest_circuit = run_trotter(m_precision, trotter_order)
        rows.append({
            "m_precision": m_precision,
            "trotter_order": trotter_order,
            "target_accuracy": target_accuracy,
            "numQubits": logical_estimate["numQubits"],
            "rotationCount": logical_estimate["rotationCount"],
            "rotationDepth": logical_estimate["rotationDepth"],
        })

df = pd.DataFrame(rows)
df


,m_precision,trotter_order,target_accuracy,numQubits,rotationCount,rotationDepth
0,6,1,0.01,5,687793,592925
1,6,2,0.01,5,28280,24240
2,8,1,0.01,5,10766395,9281375
3,8,2,0.01,5,222376,190608
4,10,1,0.01,5,173209779,149318775
5,10,2,0.01,5,1785952,1530816


In [ ]:
from qdk.widgets import Circuit

Circuit(largest_circuit.get_qsharp_circuit())